# Training 

In [1]:
### lIBRAIRIES 

In [21]:
pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Téléchargement du jeu de données (x) et y la target à prédire 

In [22]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 
  
# data (as pandas dataframes) 
x = online_retail.data.features 
y = online_retail.data.targets 
  
# metadata 
print(online_retail.metadata) 
  
# variable information 
print(online_retail.variables) 


{'uci_id': 352, 'name': 'Online Retail', 'repository_url': 'https://archive.ics.uci.edu/dataset/352/online+retail', 'data_url': 'https://archive.ics.uci.edu/static/public/352/data.csv', 'abstract': 'This is a transactional data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.', 'area': 'Business', 'tasks': ['Classification', 'Clustering'], 'characteristics': ['Multivariate', 'Sequential', 'Time-Series'], 'num_instances': 541909, 'num_features': 6, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': None, 'index_col': ['InvoiceNo', 'StockCode'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2015, 'last_updated': 'Mon Oct 21 2024', 'dataset_doi': '10.24432/C5BW33', 'creators': ['Daqing Chen'], 'intro_paper': {'ID': 361, 'type': 'NATIVE', 'title': 'Data mining for the online retail industry: A case study of RFM model-based customer segmenta

### Aperçu du dataframe et des colonnes de variables 

In [23]:
x.head()

,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [24]:
x.dtypes

Description     object
Quantity         int64
InvoiceDate     object
UnitPrice      float64
CustomerID     float64
Country         object
dtype: object

###### Note : on va garder seulement les premières 1000 lignes pour alléger la mémoire vu que c'est de l'entrainement 

In [25]:
x=x[:1000]
x.describe()

,Quantity,UnitPrice,CustomerID
count,1000.000000,1000.000000,999.000000
mean,12.785000,3.037110,16023.130130
std,38.423706,5.896942,1865.129933
min,-24.000000,0.000000,12431.000000
25%,2.000000,1.250000,14688.000000
50%,4.000000,2.100000,16210.000000
75%,12.000000,3.750000,17908.000000
max,600.000000,165.000000,18085.000000


### Traitement des données (NA,etc...)

In [26]:
x.isna().sum()

Description    1
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     1
Country        0
dtype: int64

###### Afficher la ligne ou il y a une valeur manquante 

In [27]:
x[x.isna().any(axis=1)]

,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,NaN,56,12/1/2010 11:52,0.0,NaN,United Kingdom


###### Note : on pourrait vouloir lui ajouter une description de consommation ainsi qu'un ID customer mais cela implique un biais si on utilise l'aléatoire pour l'ID par exemple donc on va simplement supprimer la ligne 

In [28]:
x.drop(index=622,inplace=True)
x.describe()

C:\Users\thoma\AppData\Local\Temp\ipykernel_21060\974857205.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  x.drop(index=622,inplace=True)


,Quantity,UnitPrice,CustomerID
count,999.000000,999.000000,999.000000
mean,12.741742,3.040150,16023.130130
std,38.418581,5.899112,1865.129933
min,-24.000000,0.100000,12431.000000
25%,2.000000,1.250000,14688.000000
50%,4.000000,2.100000,16210.000000
75%,12.000000,3.750000,17908.000000
max,600.000000,165.000000,18085.000000


In [35]:
x.tail()

,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
995,HEART OF WICKER SMALL,1,12/1/2010 12:43,1.65,14729.0,United Kingdom
996,SKULLS SQUARE TISSUE BOX,1,12/1/2010 12:43,1.25,14729.0,United Kingdom
997,PINK PAISLEY SQUARE TISSUE BOX,1,12/1/2010 12:43,1.25,14729.0,United Kingdom
998,PACK OF 6 HANDBAG GIFT BOXES,1,12/1/2010 12:43,2.55,14729.0,United Kingdom
999,TOAST ITS - HAPPY BIRTHDAY,2,12/1/2010 12:43,1.25,14729.0,United Kingdom


# Mission que l'on s'octroie 

### Faire des stats descriptives 
### Créer une variable CA qui somme tous les achats par jour et utiliser les séries temporelles pour prédire le CA d'un jour aléatoire 
### Utiliser les Kmeans pour segmenter les clients 
### Apprendre à utiliser l'ANOMALY DETECTION avec Isolation Forest/ LOF 

In [34]:
print(x[x['CustomerID']==17850.0])

                             Description  Quantity      InvoiceDate  \
0     WHITE HANGING HEART T-LIGHT HOLDER         6   12/1/2010 8:26   
1                    WHITE METAL LANTERN         6   12/1/2010 8:26   
2         CREAM CUPID HEARTS COAT HANGER         8   12/1/2010 8:26   
3    KNITTED UNION FLAG HOT WATER BOTTLE         6   12/1/2010 8:26   
4         RED WOOLLY HOTTIE WHITE HEART.         6   12/1/2010 8:26   
..                                   ...       ...              ...   
430         SET 7 BABUSHKA NESTING BOXES         2  12/1/2010 11:33   
431             IVORY EMBROIDERED QUILT          2  12/1/2010 11:33   
432    GLASS STAR FROSTED T-LIGHT HOLDER         6  12/1/2010 11:33   
433            HAND WARMER RED POLKA DOT         6  12/1/2010 11:34   
434               HAND WARMER UNION JACK         6  12/1/2010 11:34   

     UnitPrice  CustomerID         Country  
0         2.55     17850.0  United Kingdom  
1         3.39     17850.0  United Kingdom  
2         2.